# Notebook 02 · Una GAN condicional (cGAN): generar el dígito que queramos

**Introducción al Deep Learning — Módulo 3, Unidad 3: GANs (Parte II)**

La DGAN del Notebook 01 genera dígitos **al azar**: no podemos elegir cuál. La **GAN condicional (cGAN)**
resuelve esto añadiendo la **etiqueta** como entrada a las dos redes:

- El **generador** recibe ruido **+ la etiqueta** y aprende a producir esa clase.
- El **discriminador** recibe la imagen **+ la etiqueta** y rechaza los pares que no coinciden.

Al final podremos pedirle: *"genera un 7"*.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos (imágenes y sus etiquetas)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
tf.random.set_seed(42); np.random.seed(42)

d = load_digits()
X = (d.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0)
Y = d.target.astype("int32")          # ahora SÍ usamos las etiquetas
N_CLASES, DIM_RUIDO = 10, 16
print("Imágenes:", X.shape, "| Etiquetas:", Y.shape)

## 2. El generador condicional

La etiqueta pasa por una capa **Embedding** y se concatena con el ruido. Así el generador sabe QUÉ debe crear.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Flatten, Concatenate, Dense, Reshape,
                                     Conv2DTranspose, Conv2D, LeakyReLU, BatchNormalization)

# --- Generador: (ruido, etiqueta) -> imagen ---
z_in = Input(shape=(DIM_RUIDO,))
lab_in = Input(shape=(1,), dtype="int32")
lab_emb = Flatten()(Embedding(N_CLASES, 16)(lab_in))     # etiqueta -> vector
x = Concatenate()([z_in, lab_emb])                        # ruido + etiqueta
x = Dense(2 * 2 * 32)(x)
x = Reshape((2, 2, 32))(x)
x = Conv2DTranspose(32, 3, strides=2, padding="same")(x); x = BatchNormalization()(x); x = LeakyReLU(0.2)(x)
x = Conv2DTranspose(16, 3, strides=2, padding="same")(x); x = BatchNormalization()(x); x = LeakyReLU(0.2)(x)
img_out = Conv2D(1, 3, padding="same", activation="sigmoid")(x)
generador = Model([z_in, lab_in], img_out, name="generador_condicional")
print("Salida del generador:", generador.output_shape)

## 3. El discriminador condicional

Recibe la imagen **y** la etiqueta: aprende a aceptar solo los pares imagen-etiqueta **correctos y reales**.

In [ ]:
img_in = Input(shape=(8, 8, 1))
lab_in2 = Input(shape=(1,), dtype="int32")
lab_map = Embedding(N_CLASES, 8 * 8)(lab_in2)
lab_map = Reshape((8, 8, 1))(lab_map)                    # etiqueta como un "canal" extra
x = Concatenate()([img_in, lab_map])                     # imagen + etiqueta
x = Conv2D(16, 3, strides=2, padding="same")(x); x = LeakyReLU(0.2)(x)
x = Conv2D(32, 3, strides=2, padding="same")(x); x = LeakyReLU(0.2)(x)
x = Flatten()(x)
val_out = Dense(1, activation="sigmoid")(x)
discriminador = Model([img_in, lab_in2], val_out, name="discriminador_condicional")

bce = tf.keras.losses.BinaryCrossentropy()
opt_g = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
opt_d = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
print("Discriminador listo.")

## 4. Entrenar (bucle adversario condicionado)

In [ ]:
@tf.function
def paso(reales, etiquetas):
    n = tf.shape(reales)[0]
    z = tf.random.normal((n, DIM_RUIDO))
    with tf.GradientTape() as td:
        falsas = generador([z, etiquetas], training=True)
        d_real = discriminador([reales, etiquetas], training=True)
        d_fake = discriminador([falsas, etiquetas], training=True)
        loss_d = bce(tf.ones_like(d_real), d_real) + bce(tf.zeros_like(d_fake), d_fake)
    opt_d.apply_gradients(zip(td.gradient(loss_d, discriminador.trainable_variables),
                             discriminador.trainable_variables))
    z = tf.random.normal((n, DIM_RUIDO))
    with tf.GradientTape() as tg:
        falsas = generador([z, etiquetas], training=True)
        d_fake = discriminador([falsas, etiquetas], training=True)
        loss_g = bce(tf.ones_like(d_fake), d_fake)
    opt_g.apply_gradients(zip(tg.gradient(loss_g, generador.trainable_variables),
                             generador.trainable_variables))
    return loss_d, loss_g

EPOCAS, BATCH = 60, 64
for e in range(EPOCAS):
    idx = np.random.permutation(len(X))
    for i in range(0, len(X), BATCH):
        b = idx[i:i+BATCH]
        ld, lg = paso(X[b], Y[b].reshape(-1, 1))
    if e % 15 == 0:
        print(f"época {e:2d} | loss_D {ld.numpy():.3f} | loss_G {lg.numpy():.3f}")
print("Entrenamiento terminado.")

## 5. Pedirle un dígito concreto

Ahora sí: le decimos **qué clase** queremos y la genera.

In [ ]:
def generar(digito, n=6):
    z = tf.random.normal((n, DIM_RUIDO))
    # ojo: ambas entradas deben ser tensores (no mezclar tensor con array de NumPy)
    etiquetas = tf.constant(np.full((n, 1), digito, dtype="int32"))
    return generador([z, etiquetas], training=False).numpy()

for digito in [0, 3, 7]:
    ims = generar(digito)
    fig, axes = plt.subplots(1, 6, figsize=(9, 1.5))
    for ax, im in zip(axes, ims):
        ax.imshow(im.reshape(8, 8), cmap="gray"); ax.axis("off")
    fig.suptitle(f'Generados pidiendo el dígito "{digito}"', y=1.15)
    plt.show()

Cada fila corresponde a la clase que le hemos **pedido**: la cGAN nos da **control** sobre lo que genera.
Con más épocas los dígitos se reconocen mucho mejor.

## Experimenta tú

1. Sube `EPOCAS` (p. ej. 200) y compara la calidad.
2. Genera todos los dígitos del 0 al 9 en una rejilla.
3. Prueba con MNIST (28×28) en Colab con GPU.

## Resumen

- La **cGAN** añade la **etiqueta** como entrada al generador y al discriminador (capa `Embedding`).
- El generador aprende a producir cada clase; el discriminador rechaza pares imagen-etiqueta incoherentes.
- Resultado: **control** sobre lo que se genera, algo imposible con una DGAN normal.

Con esto cerramos la unidad de GANs.